In [1]:
import torch
torch.cuda.get_device_name()

AssertionError: Torch not compiled with CUDA enabled

# Практический кейс. Офлайн-генерация описаний товаров
Представьте, что вы — специалист по DS в маркетплейсе «Практикум». Как и другие маркетплейсы, «Практикум» продаёт товары из разных магазинов. 

У каждого товара есть карточка, она заполняется информацией от продавцов.

Есть исследования, что, если в карточке есть описание товара, его покупают на X% чаще. Продавцы не всегда дают такую информацию, и описания часто пишут копирайтеры из маркетплейса. Но ассортимент резко расширяется и копирайтеры не справляются с новой нагрузкой. Решено автоматизировать процесс. 

## Часть 1. Создание MVP генерации описаний

Сгенерируйте описания товаров на основе характеристик, которые дают продавцы.

У вас есть датасет с характеристиками товаров. В этом задании используйте только 100 описаний. Код для их получения:

In [2]:
import datasets
from datasets import load_dataset
import random
import pandas as pd

# Фиксируем seed
SEED = 42
random.seed(SEED)

# Загружаем датасет
dataset = load_dataset("UniqueData/asos-e-commerce-dataset")

# Выбираем split (обычно 'train')
data = dataset['train']  # или другой доступный split

# Выбираем 100 случайных примеров
random_indices = random.sample(range(len(data)), 100)
random_samples = data.select(random_indices)

target_features = ["name", "size", "category", "price", "color"]
result_data = []
for sample in random_samples:
  target_features_sample = {feature_name: sample.get(feature_name) for feature_name in target_features}
  result_data.append(target_features_sample)

df = pd.DataFrame(result_data)
df.head()

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 6/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 30845/30845 [00:00<00:00, 62826.52 examples/s]


,name,size,category,price,color
0,ASOS DESIGN Curve wrap bodysuit with angel sle...,"UK 16,UK 18,UK 20,UK 22,UK 24,UK 26,UK 28,UK 30",ASOS DESIGN Curve wrap bodysuit with angel sle...,25.00,Black
1,Bershka corset detail roll neck jumper in black,"XS - UK 6,S - UK 8,M - UK 10,L - UK 12,XL - UK...",Bershka corset detail roll neck jumper in black,Now 18.50,BLACK
2,Stradivarius oversized faux leather padded puf...,"XS - UK 6,S - UK 8,M - UK 10,L - UK 12,XL - UK 14",Stradivarius oversized faux leather padded puf...,59.99,ECRU
3,New Balance Running 1/2 zip long sleeve top in...,"XS - UK 4-6,S - UK 8-10,M - UK 12-14,L - UK 16...",New Balance Running 1/2 zip long sleeve top in...,40.00,PURPLE
4,ASYOU satin square neck cami dress with diaman...,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14,UK 16,UK 18",ASYOU satin square neck cami dress with diaman...,42.99,Black


У отдела контента есть набор требований, которым должны удовлетворять описания: 
- Описание должно начинаться с заголовка, выделенного html-тегами \<h1> и \</h1>.
- Под каждую характеристику нужно выделить отдельный абзац.
- Каждый абзац нужно выделить html-тегами \<p> и \</p>.
- В описании нужно учесть все характеристики.
- Язык должен быть разнообразным. Используется простая метрика лексического разнообразия VocD.

Вам нужно сгенерировать описания, которые будут максимально удовлетворять этим требованиям. Оценить соответствие требованиям помогут функции:

In [2]:
import re
from math import log
from collections import Counter
from difflib import SequenceMatcher

def check_h1_tags(description):
    """
    Проверяет наличие ровно одного непустого заголовка в тегах <h1>
    Возвращает 1 — если есть, 0 — если нет
    """
    h1_pattern = r'<h1>(.*?)</h1>'
    matches = re.findall(h1_pattern, description, re.DOTALL | re.IGNORECASE)

    # Должен быть ровно один непустой заголовок
    if len(matches) != 1:
        return 0

    # Проверяем, что заголовок не пустой и не состоит только из пробелов
    h1_content = matches[0].strip()
    if not h1_content or len(h1_content) < 3:
        return 0

    return 1

def similarity_ratio(a, b):
    """Вычисляет коэффициент схожести строк (0.0-1.0)"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def calculate_lexical_diversity(text):
    """
    Вычисляет лексическое разнообразие с учётом уникальности слов
    и штрафует за повторяющиеся последовательности
    """
    # Удаляем HTML-теги
    clean_text = re.sub(r'<[^>]+>', '', text)
    words = re.findall(r'\b\w+\b', clean_text.lower())

    if len(words) < 5:  # Минимальное количество слов для анализа
        return 0.0

    # Подсчитываем частоту слов
    word_counts = Counter(words)
    total_words = len(words)
    unique_words = len(word_counts)

    # Базовая метрика разнообразия
    base_diversity = unique_words / total_words

    # Штраф за повторяющиеся последовательности
    repeat_penalty = 1.0
    for i in range(len(words) - 3):
        sequence = ' '.join(words[i:i+3])
        if sequence in ' '.join(words[i+3:]):
            repeat_penalty *= 0.7  # Штраф за повторение последовательностей

    # Комбинированная метрика
    diversity = base_diversity * repeat_penalty
    return min(diversity, 1.0)

def check_paragraph_formatting(description):
    """
    Строгая проверка форматирования абзацев
    """
    p_pattern = r'<p>(.*?)</p>'
    paragraphs = re.findall(p_pattern, description, re.DOTALL | re.IGNORECASE)

    if not paragraphs:
        return 0.0

    valid_count = 0
    for paragraph in paragraphs:
        # Убираем теги и проверяем содержание
        clean_content = re.sub(r'<[^>]+>', '', paragraph).strip()

        # Абзац должен содержать значимый текст (не только теги)
        if (len(clean_content) >= 10 and  # Минимальная длина
            not clean_content.isdigit() and  # Не только цифры
            len(set(clean_content.split())) >= 3):  # Минимум 3 уникальных слова
            valid_count += 1

    return valid_count / len(paragraphs) if paragraphs else 0.0

def is_characteristic_present(text, char_name, char_value, threshold=0.6):
    """
    Более строгая проверка наличия характеристики
    """
    text_lower = text.lower()
    char_name_lower = char_name.lower()
    char_value_lower = str(char_value).lower()

    # Проверяем полное совпадение или значимое частичное
    if (char_name_lower in text_lower and char_value_lower in text_lower):
        return True

    # Проверяем семантическую близость с более высоким порогом
    name_similarity = similarity_ratio(char_name_lower, text_lower)
    value_similarity = similarity_ratio(char_value_lower, text_lower)

    # Требуем высокой схожести для обеих частей характеристики
    return (name_similarity >= threshold and value_similarity >= threshold)

def check_paragraph_tags(description, characteristics, threshold=0.6):
    """
    Строгая проверка абзацев: один абзац = одна характеристика
    """
    if not characteristics:
        return 1.0

    p_pattern = r'<p>(.*?)</p>'
    paragraphs = re.findall(p_pattern, description, re.DOTALL | re.IGNORECASE)

    if not paragraphs:
        return 0.0

    # Штрафуем за дубликаты абзацев
    unique_paragraphs = set()
    duplicate_penalty = 1.0

    for paragraph in paragraphs:
        clean_text = re.sub(r'<[^>]+>', '', paragraph).strip()
        if clean_text in unique_paragraphs:
            duplicate_penalty *= 0.5  # Сильный штраф за дубликаты
        unique_paragraphs.add(clean_text)

    # Сопоставляем характеристики с абзацами
    char_matched = {char: False for char in characteristics}
    valid_matches = 0

    for paragraph in paragraphs:
        clean_text = re.sub(r'<[^>]+>', '', paragraph).strip()

        # Ищем характеристику в абзаце
        matched_chars = []
        for char_name, char_value in characteristics.items():
            if is_characteristic_present(clean_text, char_name, char_value, threshold):
                matched_chars.append(char_name)

        # Абзац должен содержать ровно одну характеристику
        if len(matched_chars) == 1 and not char_matched.get(matched_chars[0], False):
            char_matched[matched_chars[0]] = True
            valid_matches += 1

    # Учитываем штраф за дубликаты
    coverage_score = sum(1 for matched in char_matched.values() if matched) / len(characteristics)
    return coverage_score * duplicate_penalty

def check_all_characteristics_covered(description, characteristics, threshold=0.5):
    """
    Проверяет coverage характеристик с учётом уникальности
    """
    if not characteristics:
        return 1.0

    clean_description = re.sub(r'<[^>]+>', '', description)
    found_chars = set()

    for char_name, char_value in characteristics.items():
        if is_characteristic_present(clean_description, char_name, char_value, threshold):
            found_chars.add(char_name)

    return len(found_chars) / len(characteristics)

def check_text_quality(description):
    """
    Новая функция: проверяет общее качество текста
    """
    clean_text = re.sub(r'<[^>]+>', '', description)
    words = clean_text.split()

    if len(words) < 10:
        return 0.0

    # Проверяем разнообразие (повторы, уникальность)
    word_counts = Counter(words)
    unique_ratio = len(word_counts) / len(words)

    # Штрафуем за слишком частые повторы
    repeat_penalty = 1.0
    for word, count in word_counts.items():
        if count > len(words) * 0.1:  # Слово повторяется более 10% текста
            repeat_penalty *= 0.7

    return min(unique_ratio * repeat_penalty, 1.0)

def calculate_total_score(description, characteristics):
    """
    Пересмотренная система оценки с более строгими критериями
    """
    # Выполняем все проверки
    h1_score = check_h1_tags(description)
    paragraph_score = check_paragraph_tags(description, characteristics, threshold=0.7)
    coverage_score = check_all_characteristics_covered(description, characteristics, threshold=0.6)
    diversity_score = calculate_lexical_diversity(description)
    formatting_score = check_paragraph_formatting(description)
    text_quality_score = check_text_quality(description)

    # Новые весовые коэффициенты (увеличиваем вес за качество)
    weights = {
        'h1': 0.10,
        'paragraphs': 0.25,
        'coverage': 0.15,
        'diversity': 0.20,
        'formatting': 0.15,
        'quality': 0.15  # Новый критерий качества текста
    }

    total_score = (
        h1_score * weights['h1'] +
        paragraph_score * weights['paragraphs'] +
        coverage_score * weights['coverage'] +
        diversity_score * weights['diversity'] +
        formatting_score * weights['formatting'] +
        text_quality_score * weights['quality']
    ) * 100

    return {
        'total_score': round(total_score, 2),
        'detailed_scores': {
            'h1_present': h1_score,
            'paragraphs_validation': round(paragraph_score, 2),
            'characteristics_coverage': round(coverage_score, 2),
            'lexical_diversity': round(diversity_score, 2),
            'paragraph_formatting': round(formatting_score, 2),
            'text_quality': round(text_quality_score, 2)
        }
    }

# Пример использования
if __name__ == "__main__":
    test_description = """
    <h1>Смартфон Premium X</h1>
    <p>Данная модель обладает превосходным дисплеем размером 6.7 дюймов.</p>
    <p>Основная камера 108 мегапикселей обеспечивает отличное качество фото.
    <p>Процессор Snapdragon 8 второго поколения обеспечивает высокую производительность.</p>
    """

    test_characteristics = {
        'экран': '6.7 дюймов',
        'камера': '108 Мп',
        'батарея': '5000 мАч',
        'процессор': 'Snapdragon 8 Gen 2'
    }

    result = calculate_total_score(test_description, test_characteristics)
    print("Общий скор:", result['total_score'])
    print("Детальные результаты:", result['detailed_scores'])

Общий скор: 58.73
Детальные результаты: {'h1_present': 1, 'paragraphs_validation': 0.0, 'characteristics_coverage': 0.0, 'lexical_diversity': 0.96, 'paragraph_formatting': 1.0, 'text_quality': 0.96}


Визуализировать результат помогут функции-помощники: 

In [1]:
import copy
from dataclasses import dataclass
from typing import List
import textwrap
import random

@dataclass
class SampleGenerationResult():
  description: str
  latency: float
  total_score: float
  detailed_score: dict

@dataclass
class FullGenerationResult():
  throughput: float
  latency_avg: float
  total_score_avg: float
  outputs: List[SampleGenerationResult]

def print_compact_stats(results: FullGenerationResult) -> None:
    """
    Компактная версия вывода статистики.
    """
    print(" СТАТИСТИКА ГЕНЕРАЦИИ")
    print(f"   Throughput: {results.throughput:.2f} req/sec")
    print(f"   Avg Latency: {results.latency_avg:.3f} sec")
    print(f"   Samples: {len(results.outputs)}")
    print(f"   Avg Score: {sum(s.total_score for s in results.outputs)/len(results.outputs):.3f}")

    # Добавляем случайные описания в компактную версию
    if len(results.outputs) >= 3:
        print("\n 3 случайных описания:")
        random_indices = random.sample(range(len(results.outputs)), min(3, len(results.outputs)))
        for i, idx in enumerate(random_indices, 1):
            desc = results.outputs[idx].description
            print(f"   {i}. {desc}")

### Задание 1
Напишите код для офлайн-генерации описаний на основе данных из скачанного датасета.
- Замерьте эффективность генерации.
  - Чтобы оценить качество, используйте представленные выше функции.
  - Напишите функции для измерения основных характеристик скорости инференса: latency (секунд на ответ) и throughput (ответов в секунду).
- Представьте полученные результаты по скорости и качеству инференса для датасета в виде экземпляра класса `FullGenerationResult()`. Используйте функции-помощники, представленные выше.
- Подберите такой промпт для модели, чтобы в среднем получить качество выше порогового значения `result['total_score']>50` по совокупной метрике `total_score`.

Сначала загрузите нужную модель:

In [1]:
import gc
import torch
from vllm import LLM
from string import Template

# Удаление модели и освобождение памяти
# del llm
gc.collect()

# Дополнительная очистка GPU памяти (если используется CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

model_name = "Qwen/Qwen3-30B-A3B-Instruct-2507"

# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model=model_name,
    tensor_parallel_size=1,
    max_model_len=20_000
)

/home/ubuntu/project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-19 15:08:05,239	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Загрузка модели...


ValueError: The checkpoint you are trying to load has model type `qwen3_moe` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

In [ ]:
from string import Template


# придумайте шаблон промпта, который можно будет заполнить
# разными входными характеристиками для разных товаров
prompt_template = Template("""
Создай подробное и привлекательное описание товара для интернет‑магазина на основе следующих характеристик: name, size, category, price, color. Соблюдай структуру и правила ниже:
Структура описания:
- В самом начале создай заголовок — это должно быть краткое и ёмкое название товара. Оформи его с помощью HTML‑тегов: <h1>Заголовок</h1>. Используй в заголовке ключевые слова из поля name (например, тип изделия, бренд, особенности дизайна), но сделай формулировку более маркетинговой и читабельной — убери лишние детали и повторы.
- Далее добавь несколько абзацев с описанием характеристик. Каждый абзац должен:
  - начинаться с краткого подзаголовка‑маркера (жирным шрифтом), описывающего характеристику (например, Размер, Категория, Цена, Цвет);
  - содержать развёрнутое описание значения из соответствующего поля данных — не просто копируй значение, а перепиши его естественным, продающим языком;
  - быть заключён в HTML‑теги <p>...</p>.
- В последнем абзаце (тоже в тегах <p>...</p>) добавь короткий призыв к действию — мотивируй покупателя приобрести товар. Используй информацию из описания, чтобы сделать призыв релевантным (например, упомяни универсальность цвета или выгодность цены).

Правила и рекомендации:
- Язык описания — русский, стилистика — продающая, но без излишнего пафоса. Тон дружелюбный и доверительный.
- Избегай повторов слов и конструкций. Используй разнообразную лексику: подбирай синонимы, варьируй длину предложений.
- Для характеристики size представь размеры в виде списка (если их несколько), используя маркированный список в HTML (<ul><li>...</li></ul> внутри тега <p>).
- Для price укажи цену с валютой (предполагается, что цена в фунтах стерлингов — GBP). Оформи числовое значение так, чтобы оно хорошо читалось (например, 25,00 GBP).
- В описании color не просто назови цвет, а добавь пару слов о его универсальности, сочетаемости или настроении, которое он создаёт.
- Учитывай все предоставленные характеристики — ни одна не должна остаться без абзаца.
- Общий объём описания — 100–200 слов.

Формат вывода:
- только готовый HTML‑код описания, без каких‑либо дополнительных комментариев, пояснений или текста до/после кода.

Данные:
- Название: $name
- Размер: $size
- Категория: $category
- Цена: $price
- Цвет: $color
""")

In [3]:
from vllm import SamplingParams
import time

# Создайте список итоговых промптов, которые хотите отправить по всем 100 товарам
input_data = [
  prompt_template.substitute(
    name=name,
    size=size,
    category=category,
    price=price,
    color=color
  )
  for name, size, category, price, color in zip(
    df['name'], df['size'], df['category'], df['price'], df['color'])
]

# Задайте параметры семплирования
sampling_params = SamplingParams(temperature=0.1,
                                top_p=0.9,
                                max_tokens=600,
                                presence_penalty=1.2,
                                repetition_penalty=1,
                                )

def run_generation(input_data, sampling_params, llm):
  start_time = time.time()
  generated_data = llm.generate(input_data, sampling_params)
  end_time = time.time()

  latency = end_time - start_time
  throughput = len(generated_data) / latency if latency > 0 else 0

  # В этом случае все ответы мы получаем одновременно,
  # Поэтому можно сказать, что время каждого ответа равно времени полной генерации всех ответов
  # Это основной недостаток офлайн-генерации
  latency_list = [latency for _ in range(len(generated_data))]

  return generated_data, throughput, latency_list


def prepare_results(generated_data, throughput, latency_list):
  generation_results = []
  for output, features, latency in zip(generated_data, result_data, latency_list):
    description = output.outputs[0].text
    score_info = calculate_total_score(output.outputs[0].text, features)

    sample_info = SampleGenerationResult(
        latency=latency,
        description=description,
        total_score=score_info['total_score'],
        detailed_score=copy.deepcopy(score_info['detailed_scores'])
    )
    generation_results.append(sample_info)


  full_generation_result = FullGenerationResult(
      throughput=throughput,
      # реализуйте получение среднего время ответа
      latency_avg=sum([x.latency for x in generation_results])/len(generation_results),
      # реализуйте получение среднего скора
      total_score_avg=sum([x.total_score for x in generation_results])/len(generation_results),
      outputs=generation_results
  )
  return full_generation_result



generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")


full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

ModuleNotFoundError: No module named 'vllm'

## Часть 2. Повышение эффективности генерации

Представим, что в маркетплейсе планируются сезонные скидки для продавцов и прогнозируется, что в ближайший месяц продавцы выставят в 2 раза больше товаров. А значит, нужно сгенерировать больше описаний.

Ваша задача — повысить throughput, проседая не более 5% по имеющейся метрике. Сравните два способа оптимизации эффективности: квантизацию и спекулятивный декодинг. Выберите тот, который, на ваш взгляд, в этой задаче более уместен.


<div style="background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 10px; margin: 10px 0;">

В установленной версии VLLM не поддерживается спекулятивный декодинг с драфт-моделью, с которым вы познакомились в этой теме.

Вместо него используйте спекулятивный декодинг на n-граммах из промпта.

Он использует статистику n-граммов токенов из промпта как простейшую драфт языковую модель. 
</div>

Подробнее — [в документации](https://docs.vllm.ai/en/v0.5.3/models/spec_decode.html).


Пример запуска офлайн-генерации со спекулятивным декодингом:

In [ ]:
# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model=<model_name>,
    tensor_parallel_size=1,
    max_model_len=20_000,
    speculative_config={
        "method": "ngram",
        "num_speculative_tokens": ...,
        "prompt_lookup_max": ...,
    },
)

Пример запуска модели с квантизацией

In [ ]:
quantized_model_name = ...

# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model=quantized_model_name,
    tensor_parallel_size=1,
    max_model_len=20_000
)

### Задание 2

- Чтобы запустить квантизированную модель, самостоятельно найдите на HF квантизированную версию модели `Qwen/Qwen3-30B-A3B-Instruct-2507`.
- Чтобы запустить спекулятивный декодинг, подберите подходящие параметры `num_speculative_tokens`, `prompt_lookup_max`. Объясните свой выбор.
- Проведите генерацию с используемыми моделями. Сравните результаты по качеству и скорости и пропускной способности.
- Сделайте выводы о применимости методов к этой задаче. Ответьте на вопросы:
  - Как изменилось качество при спекулятивном декодинге? Почему?
  - Как изменилось качество при квантизации? Почему?
  - Как изменилась скорость генерации при спекулятивном декодинге? Почему?
  - Как изменилась скорость генерации при квантизации? Почему?

---

Чтобы найти преобразования над определённой моделью на HF, можно воспользоваться [деревом карточек модели](https://huggingface.co/models?other=base_model:Qwen%2FQwen3-30B-A3B-Instruct-2507&sort=trending).

In [ ]:
##-----------------------------------------------------------------------------------
## для инференса со спекулятивным декодингом

# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model="Qwen/Qwen3-30B-A3B-Instruct-2507",
    tensor_parallel_size=1,
    max_model_len=20_000,
    speculative_config={
        "method": "ngram",
        "num_speculative_tokens": 5,
        "prompt_lookup_max": 4,
    },
)

generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")

full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

##-----------------------------------------------------------------------------------

## для инференса с квантизацией
model_name = "cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit"

# Инициализация модели
print("Загрузка модели...")
llm = LLM(
    model=model_name,
    tensor_parallel_size=1,
    max_model_len=20_000
)

generated_data, throughput, latency_list = run_generation(input_data, sampling_params, llm)
print(f"Длительность ответа на единичный запрос {sum(latency_list)/len(latency_list)}")
print(f"Количество запросов, обрабатываемых в секунду {throughput}")

full_generation_result = prepare_results(generated_data, throughput, latency_list)
print_compact_stats(full_generation_result)

##-----------------------------------------------------------------------------------

Итак, вы попробовали использовать квантизацию и спекулятивный декодинг на практике для офлайн-инференса. Сравнили эффективность методов как по качеству, так и по скорости на задаче генерации описаний в офлайне. Но что, если нужно как можно быстрее отвечать на каждый отдельный запрос?

## Онлайн-генерация
Онлайн-генерация поможет значительно снизить время ответа на единичный запрос и сделать сервис, с которым могут взаимодействовать внутренние сотрудники.

Например, копирайтеры маркетплейса могут генерировать описания, чтобы получить черновую версию, которую они затем скорректируют до итогового варианта.

Сначала запустим онлайн-инференс, используя VLLM. Предварительно удалите предыдущую модель и очистите память. Для наивного запуска онлайн VLLM сервиса на основе Qwen/Qwen3-30B-A3B-Instruct-2507 откройте терминал и выполните в нём команду:

In [ ]:
vllm serve "Qwen/Qwen3-30B-A3B-Instruct-2507" \
  --dtype auto \
  --api-key token-abc123 \
  --max-model-len '20k'

Теперь посмотрим, как отправлять запросы для онлайн-обработки в LLM. 

В следующих двух заданиях мы симулируем поведение, когда запросы приходят в сервис из нескольких независимых источников и параллельно обрабатываются. Для этого используем асинхронную отправку запросов. Фрагмент кода ниже реализует все вспомогательные методы для симуляции нагрузки на онлайн-сервис генерации:

In [ ]:
import aiohttp
import asyncio
import time
import nest_asyncio

nest_asyncio.apply()

async def send_to_generation(prompt, model_name="Qwen/Qwen3-30B-A3B-Instruct-2507", sampling_params=None):
    base_url = "http://localhost:8000/v1"
    params = copy.deepcopy(sampling_params) or {}
    params["model_name"] = model_name
    params["prompt"] = prompt
    async with aiohttp.ClientSession() as session:
        start_time = time.time()
        print(f"start {time.strftime('%H:%M:%S')} - {prompt[:20]}...")
        async with session.post(
            f"{base_url}/completions",
            json={
                "model": model_name,
                "prompt": prompt,
                "max_tokens": 500,
                "temperature": 0.7
            },
            headers={"Authorization": "Bearer token-abc123"},
            timeout=100
        ) as response:
            result = await response.json()
            end_time = time.time()
            print(f"end {time.strftime('%H:%M:%S')} - {prompt[:20]}... (duration: {end_time-start_time:.5f}s)")
            return result, start_time, end_time

async def schedule_requests(prompts, pause_duration=1, sampling_params=None, model_name="Qwen/Qwen3-30B-A3B-Instruct-2507"):
    tasks = []

    for i, prompt in enumerate(prompts):
        # Создаём задачу, но не ждём её завершения
        task = asyncio.create_task(send_to_generation(prompt, model_name, sampling_params))
        tasks.append(task)

        # Ждём pause_duration секунд перед следующим запросом
        if i < len(prompts) - 1:  # Не ждём после последнего запроса
            await asyncio.sleep(pause_duration)

    # Ждём завершения всех задач
    results = await asyncio.gather(*tasks)
    return results

# Запуск
prompts = [
    "Качественный сервис это",
    "Объясни понятие 'машинное обучение' простыми словами:",
    "Напиши рецепт быстрого ужина из курицы:",
    "Какие преимущества у использования Python для data science?"
    "Напиши определение объектно ориентированного программирования",
]
results = await schedule_requests(prompts, 2)

Функция `send_to_generation` отправляет одиночный асинхронный запрос в наш сервис онлайн-инференса на VLLM. В ответ отдаёт объект с генерацией от VLLM — result, время отправки запроса — `start_time`, время получения ответа — `end_time`.

Поведение копирайтеров, которые независимо отправляют запросы в сервис генерации, реализуем упрощённо. Представим, что в среднем каждые `n` секунд отправляется новый запрос на генерацию. Моделировать такое поведение поможет функция `schedule_requests`.

Обратите внимание на вывод запуска ячейки со вспомогательными методами, приведённый ниже:

In [ ]:
start 20:22:33 - Качественный сервис ...
start 20:22:35 - Объясни понятие 'маш...
start 20:22:37 - Напиши рецепт быстро...
end 20:22:38 - Качественный сервис ... (duration: 5.60702s)
start 20:22:39 - Какие преимущества у...
end 20:22:42 - Объясни понятие 'маш... (duration: 6.91522s)
end 20:22:44 - Напиши рецепт быстро... (duration: 6.89132s)
end 20:22:45 - Какие преимущества у... (duration: 6.13210s)

Заметно, что запросы обрабатываются независимо и ответ на первый приходит до отправки четвёртого. Это достигается за счёт непрерывного батчинга.

Вы познакомились с основными инструментами, переходим к практике. 


## Часть 3. Анализ онлайн-инференса
Ранее мы отправляли все запросы разом. Теперь у нас появляется ещё одна степень свободы — нагрузка на сервис, то есть количество запросов, отправляемых в сервис в единицу времени.

Посмотрим на поведение сервиса под разной нагрузкой. 

### Задание 3
1. Напишите функцию, которая:
   - принимает на вход результаты обработки запросов на генерацию описаний (`input_data` из первого задания) функцией `schedule_requests`
   - и в ответ выдаёт экземпляр класса `FullGenerationResult` из первого задания с полностью заполненными атрибутами.
1. Получите результаты для разных периодов отправки запросов `pause_duration=4,2,1,0.5`. В процессе понаблюдайте за характеристиками `Running: x reqs, Waiting: x reqs` в терминале с запущенным VLLM-сервером при генерации текста.
1. Объясните полученные результаты. Ответьте на вопросы:
   - Как меняется время на единичный запрос и пропускная способность с изменением нагрузки на сервис? Почему?
   - Что произойдёт, если нагрузка на сервис повысится многократно и выйдет за рамки практической реализации в задании?
   - Как меняется качество с изменением нагрузки на сервис? Почему?

In [2]:
def prepare_results_online(results):
    generated_data = [res[0] for res in results]
     # вычислите latency для каждого запроса
    latency_list = [res[2] - res[1] for res in results]
    # вычислите пропускную способность
    throughput = len(generated_data) / sum(latency_list) if sum(latency_list) > 0 else 0 

    generation_results = []
    for output, features, latency in zip(generated_data, result_data, latency_list):
        description = output['choices'][0]['text']
        score_info = calculate_total_score(output['choices'][0]['text'], features)

        sample_info = SampleGenerationResult(
            latency=latency,
            description=description,
            total_score=score_info['total_score'],
            detailed_score=copy.deepcopy(score_info['detailed_scores'])
        )
        generation_results.append(sample_info)

    full_generation_result = FullGenerationResult(
        throughput=throughput,
        latency_avg=sum([x.latency for x in generation_results])/len(generation_results),
        total_score_avg=sum([x.total_score for x in generation_results])/len(generation_results),
        outputs=generation_results
    )
    return full_generation_result

##-----------------------------------------------------------------------
# Получите результаты для разных нагрузок из условия
results = await schedule_requests(input_data, pause_duration=...)
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

##-----------------------------------------------------------------------
# Ответьте на вопросы:
#- Как меняется время на единичный запрос и пропускная способность с изменением нагрузки на сервис? Почему?
#- Что произойдёт, если нагрузка на сервис повысится многократно и выйдет за рамки практической реализации в задании?
#- Как меняется качество с изменением нагрузки на сервис? Почему?


NameError: name 'schedule_requests' is not defined

#### Выводы:
Когда растёт нагрузка на сервис, повышается его пропускная способность и увеличивается время на обработку единичного запроса. Это связано с тем, что обработка запросов становится параллельной, к тому же появляются задержки на добавление текстов в непрерывный батч.

Когда нагрузка повысится критически и превысит максимальную пропускную способность сервиса, запросы начнут копиться в очереди, появятся `Waiting: x reqs != 0`.

Если нагрузка меняется, качество должно сохраняться. Несмотря на то что запросы генерируются в батче, генерация каждого независима от других.

В этом задании вы познакомились с онлайн-инференсом на конкретном примере генерации описаний на основе атрибутов товаров. Теперь оптимизируем онлайн-инференс. 


## Часть 4. Оптимизация онлайн-инференса

Для оптимизации инференса снова обратимся к квантизации. Посмотрим, как меняется поведение модели при изменении нагрузки на сервис. 

## Задание 4
- Запустите онлайн-инференс VLLM квантизированной модели, которую вы подобрали в задании 2.
- По аналогии с заданием 3 получите результаты для разных периодов отправки запросов `pause_duration=4,2,1,0.5`.
- Объясните полученные результаты. Ответьте на вопросы:
   - Даёт ли выигрыш по скорости или пропускной способности квантизация при малой нагрузке на сервис? Почему?
   - Даёт ли выигрыш по скорости или пропускной способности квантизация при высокой нагрузке на сервис? Почему?
   - Как вы считаете, как поведёт себя модель, если использовать спекулятивный декодинг?


In [ ]:
##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=4, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 0.25 req/sec
#    Avg Latency: 2.661 sec
#    Samples: 100
#    Avg Score: 43.244

##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=2, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 0.50 req/sec
#    Avg Latency: 2.600 sec
#    Samples: 100
#    Avg Score: 43.030

##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=1, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 0.97 req/sec
#    Avg Latency: 3.325 sec
#    Samples: 100
#    Avg Score: 44.160

##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=0.5, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 1.88 req/sec
#    Avg Latency: 3.640 sec
#    Samples: 100
#    Avg Score: 44.418

##----------------------------------------------------------------------------------------------------------
results = await schedule_requests(input_data, pause_duration=0, model_name="cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
full_generation_result = prepare_results_online(results)
print_compact_stats(full_generation_result)

#  СТАТИСТИКА ГЕНЕРАЦИИ
#    Throughput: 9.19 req/sec
#    Avg Latency: 9.741 sec
#    Samples: 100
#    Avg Score: 45.704

#### Выводы:
Квантизация даёт выигрыш по скорости как при низкой, так и при высокой нагрузке. Разница заметнее при высоких нагрузках, так как задача меньше зависит от обращений к памяти. В основном оптимизируется именно вычислительная часть.

Спекулятивный декодинг, наоборот, дал бы больше прироста в случае низкой нагрузки.